# Q2 공시 Pandas 패턴 분석 — 이정수 / Issue #7

Q2는 분석 질문 번호이며 2분기로 고정하지 않습니다.
팀 공통 DB와 설정을 읽고 월별·제목 유형별 빈도, 전월 대비 변화, 확인 후보와 민감도를 계산합니다.
실제 데이터 미제공 상태에서는 실행 결과를 만들지 않습니다. 설정·DB가 준비되면 **Run All**로 다시 실행하세요.


In [ ]:
from pathlib import Path
import sys
import os
import json

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.disclosure_pandas import run_analysis

# 자동 검색: data/processed/disclosure_config.json → config/disclosure.json
# 별도 파일이 필요하면 CONFIG에 경로를 지정하세요.
CONFIG = os.environ.get("DISCLOSURE_CONFIG")
monthly, variations, report = run_analysis(ROOT, CONFIG)
print(json.dumps(report["audit"], ensure_ascii=False, indent=2))


## 1. 집계 정책과 데이터 품질

- 접수번호가 같고 필수 컬럼 내용이 같으면 1건만 유지합니다. 상충 내용·잘못된 날짜·결측은 중단합니다.
- 정정공시는 별도 접수번호면 기본 포함합니다. 설정에서 제외할 수 있지만 원본의 최신 버전으로 대체하지는 않습니다.
- 제목에 사업보고서→반기보고서→분기보고서가 포함되는 순서로 분류하고, 나머지는 기타입니다. 공식 공시 유형 코드가 아닙니다.
- 시작 포함·종료 제외인 완전한 월을 비교합니다. 수집 담당자가 완료를 확인한 범위에만 빈 달을 0으로 채웁니다.
- 공시에는 재무제표 연결/별도 필터를 적용하지 않습니다. 팀의 기업·공시 기간 기준은 공유합니다.


In [ ]:
print(monthly.to_string(index=False))
assert int(monthly["count"].sum()) == report["audit"]["analysis_rows"]
print("집계 합계 검증 PASS")


## 2. 확인 후보와 민감도

전월 건수가 양수이면 `(현재-전월)/전월×100`을 계산합니다.
기본 후보는 **100% 이상 증가하고 3건 이상 증가**한 경우입니다. 전월 0건이면 증감률은 결측으로 두고 3건 이상 발생만 별도 후보로 표시합니다.
전월이 수집 범위 밖이면 후보 여부도 결측입니다. 감소는 이 증가 탐지 규칙의 대상이 아닙니다.
임계값은 학습된 위험 기준이 아닌 설명 가능한 임시 기준입니다. 50/100/200%와 3/5건 조합으로 민감도를 확인합니다.


In [ ]:
print(monthly.loc[monthly.candidate.fillna(False)].to_string(index=False))
print(variations.to_string(index=False))
print("확인 후보는 추가 검토 대상이며 기업 위험·부실·원인을 뜻하지 않습니다.")


## 3. SQL 대조와 팀 통합

자체 참조 SQL과 나지수님의 SQL 비교를 분리합니다. 팀 파일 `sql/queries_disclosure.sql`이 도착하면 재실행 시 자동으로 읽습니다.
팀 SQL은 단일 읽기 쿼리, 동일한 정책과 키(corp_code, period, disclosure_type), count 컬럼이 필요합니다.
파일이 없거나 형식이 다르면 CHECK로 남깁니다. 불일치 그룹을 임의로 0으로 바꾸지 않습니다.
`data/processed/disclosure_analysis`의 CSV/JSON을 최종 분석·임도윤 교차검증에서 사용할 수 있습니다.


In [ ]:
print(json.dumps({"reference_sql": report["reference_sql"], "team_sql": report["team_sql"]}, ensure_ascii=False, indent=2))
print("전체 상태:", report["status"])
print("데이터 구분:", report["dataset_kind"])
